# QA Fine-Tuner on Google Colab

Fine-tune **Phi-4-mini** or **Llama-3.2-3B** on your own question-answer CSV with QLoRA, then deploy to a HuggingFace Space.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (free tier is enough).

Steps:
1. Check the GPU
2. Get the project code
3. Install libraries (~3 min)
4. (Optional) Add your HuggingFace token and keep results on Google Drive
5. **Either** open the Gradio app (Option A) **or** run everything from cells (Option B)

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the project code

**If the repo is on GitHub:** set `REPO_URL` below. For a private repo use
`https://<your-github-token>@github.com/<org>/<repo>.git`.

**If you have the folder as a zip:** upload `P10-keerthi-main.zip` in the Files panel on the left, leave `REPO_URL` empty and run the cell.

In [ ]:
import os, zipfile, glob

REPO_URL = "https://github.com/gvpcoe-internship-2026/P10-keerthi.git"  # change if needed, or set "" to use a zip

%cd /content
if REPO_URL:
    !git clone -q {REPO_URL} project
else:
    zips = glob.glob("/content/*.zip")
    assert zips, "Upload the project zip first (Files panel on the left)."
    zipfile.ZipFile(zips[0]).extractall("/content/project_zip")
    inner = [d for d in glob.glob("/content/project_zip/*") if os.path.isdir(d)]
    os.rename(inner[0] if len(inner) == 1 else "/content/project_zip", "/content/project")
%cd /content/project
!ls

## 3. Install libraries

In [ ]:
!pip install -q -r requirements.txt

## 4. HuggingFace token and Google Drive (optional but recommended)

* **Token:** add a secret named `HF_TOKEN` (key icon on the left, *write* access token from https://huggingface.co/settings/tokens). Needed for deploying.
* **Drive:** Colab deletes files when the session ends. Saving runs and the MLflow database to Drive keeps them.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception as e:
    print("No HF_TOKEN secret found (you can paste the token in the app instead):", e)

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    base = "/content/drive/MyDrive/qa-finetune"
    os.makedirs(base, exist_ok=True)
    os.environ["QA_OUTPUT_ROOT"] = f"{base}/outputs"
    os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{base}/mlflow.db"
    print("Runs will be saved in", base)

## 5A. Option A - open the Gradio app

Run the cell and click the **public URL** (`https://....gradio.live`). Then:
1. **Data** tab - upload your CSV (or click *Use the sample dataset*)
2. **Train** tab - pick the model, press *Start training*, watch the loss curve
3. **Test** tab - ask questions
4. **Deploy** tab - push the adapter and create the Space

The cell keeps running while the app is open. Stop it (■) to continue with other cells.

In [ ]:
!python app.py --share

## 5B. Option B - command line

Same pipeline without the UI. Change `--csv` to your own file (upload it to `/content/project/data/`).

In [ ]:
!python -m qa_finetune.cli train \
    --csv data/sample_qa.csv \
    --model phi4 \
    --epochs 3 \
    --system-prompt "You are the help desk assistant of Nimbus City Library."

In [ ]:
import glob, os, json
root = os.getenv("QA_OUTPUT_ROOT", "outputs")
RUN = sorted(glob.glob(f"{root}/*/training_summary.json"), key=os.path.getmtime)[-1].rsplit("/", 1)[0]
print("Latest run:", RUN)
print(json.dumps(json.load(open(f"{RUN}/training_summary.json"))["metrics"], indent=2))

In [ ]:
!python -m qa_finetune.cli ask --run "{RUN}" "Is the library open on public holidays?"

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)
pd.read_csv(f"{RUN}/predictions.csv")

### Deploy to HuggingFace (adapter repo + Gradio Space)

In [ ]:
!python -m qa_finetune.cli deploy --run "{RUN}" --model-repo qa-qlora-adapter --space qa-qlora-chat

### MLflow experiment history

The app's **Experiments** tab shows all runs. To see them as a table here:

In [ ]:
import mlflow, os
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db"))
runs = mlflow.search_runs(experiment_names=["qa-qlora-finetune"])
cols = [c for c in runs.columns if c.startswith(("tags.mlflow.runName", "params.resolved_model_id",
        "params.learning_rate", "params.lora_r", "metrics.train_loss", "metrics.base_", "metrics.ft_"))]
runs[cols]